In [1]:
%cd /workspace/verl/verl
! pwd
import sys, os
# import current pwd to sys.path
sys.path.insert(0, os.getcwd())
sys.path.insert(0, "/workspace/verl/verl/LSTLLM")


/workspace/verl/verl
/workspace/verl/verl


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from LSTLLM.data_preprocess.raw_data_preprocesses import process_memoryagentbench_raw, process_memoryagentbench_from_memalpha

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
process_memoryagentbench_raw(
    output_path="./data",
    max_tokens_per_chunk=2048,
    split_test=0.2,
    batch_size=500,
    resume=False
    )

Processing train and test data, train output path: ./data/MemoryAgentBench_from_raw_train.parquet, test output path: ./data/MemoryAgentBench_from_raw_test.parquet
Processing split: Accurate_Retrieval


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22/22 [03:58<00:00, 10.82s/it]


Processing split: Test_Time_Learning


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:23<00:00,  3.90s/it]


Processing split: Long_Range_Understanding


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 110/110 [00:24<00:00,  4.49it/s]


Processing split: Conflict_Resolution


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:32<00:00,  4.08s/it]


'./data/MemoryAgentBench_from_raw_train.parquet'

In [9]:
import pyarrow.parquet as pq
import pandas as pd
from datasets import load_dataset
ds_train = load_dataset("parquet", data_files="/workspace/verl/verl/LSTLLM/data/MemoryAgentBench_from_raw_train.parquet")['train']
ds_test = load_dataset("parquet", data_files="/workspace/verl/verl/LSTLLM/data/MemoryAgentBench_from_raw_test.parquet")['train']

In [12]:
ds_test

Dataset({
    features: ['prompt', 'chunks', 'num_chunks', 'question', 'answers', 'target_answer', 'agent_role', 'data_source', 'sub_source', 'extra_info', 'metadata'],
    num_rows: 684
})

In [11]:
import json
# print(ds)
sample = ds_train[0]
print("\nsample keys:\n", sample.keys(), "\n", end="-"*25)
test_extrainfo = json.loads(sample["extra_info"])
print("\nextrainfo keys:\n", test_extrainfo.keys(), "\n", end="-"*25)
test_metadata = json.loads(sample["metadata"])
print("\nmetadata keys:\n", test_metadata.keys(), "\n", end="-"*25)
print("-"*25)
pre_agent_prompt = test_extrainfo["pre_agent"]
for k, v in pre_agent_prompt.items():
    print(k)
    print(v)
    print("-"*25)



sample keys:
 dict_keys(['prompt', 'chunks', 'num_chunks', 'question', 'answers', 'target_answer', 'agent_role', 'data_source', 'sub_source', 'extra_info', 'metadata']) 
-------------------------
extrainfo keys:
 dict_keys(['data_source', 'sub_source', 'pre_agent', 'sample_id', 'instance_id', 'question_idx', 'turn_roles', 'turn_metadata']) 
-------------------------
metadata keys:
 dict_keys(['demo', 'haystack_sessions', 'keypoints', 'previous_events', 'qa_pair_ids', 'question_dates', 'question_ids', 'question_types', 'source']) 
--------------------------------------------------
answer_gen
[{'role': 'system', 'content': "Generate outputs that directly address the user's needs, applying stored knowledge and contextual understanding in a coherent and practical manner."}]
-------------------------
fact_split
[{'role': 'system', 'content': 'Extract factual elements without summarization. Preserve original meaning while ensuring that each fact is isolated, concise, and relevance-scored.'}